In [1]:
import json
import time

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder

from benchmark.config import DATA_PROCESSED, DATASETS, MODELS_DIR, RESULTS_BENCHMARK
from benchmark.metrics import (
    evaluate_classification,
    plot_confusion_matrix,
    plot_roc_curve,
    save_classification_report,
)
from benchmark.models.xrfm_model import make_processor
from benchmark.models.xrfm_model import fit_best_xrfm, fit_xrfm_cross_validation

data_name = "obesity_level"
cfg = DATASETS[data_name]
processed = DATA_PROCESSED / data_name
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/shado/college_project/comp9417/9417GroupProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv(processed / "train.csv")
train_df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,26,1.628019,108.265922,yes,yes,3,3,Sometimes,no,2,no,0,0,Sometimes,Public_Transportation,Obesity_Type_III
1,Female,23,1.480000,60.000000,yes,yes,2,1,Sometimes,yes,1,no,0,0,Sometimes,Public_Transportation,Overweight_Level_II
2,Female,15,1.650000,86.000000,yes,yes,3,3,Sometimes,no,1,no,3,2,no,Walking,Obesity_Type_I
3,Female,26,1.593321,110.168166,yes,yes,3,3,Sometimes,no,2,no,0,0,Sometimes,Public_Transportation,Obesity_Type_III
4,Male,18,1.803527,108.251044,yes,yes,2,2,Sometimes,no,3,no,1,1,no,Public_Transportation,Obesity_Type_I


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1688 entries, 0 to 1687
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Gender                          1688 non-null   object 
 1   Age                             1688 non-null   int64  
 2   Height                          1688 non-null   float64
 3   Weight                          1688 non-null   float64
 4   family_history_with_overweight  1688 non-null   object 
 5   FAVC                            1688 non-null   object 
 6   FCVC                            1688 non-null   int64  
 7   NCP                             1688 non-null   int64  
 8   CAEC                            1688 non-null   object 
 9   SMOKE                           1688 non-null   object 
 10  CH2O                            1688 non-null   int64  
 11  SCC                             1688 non-null   object 
 12  FAF                             16

In [4]:
test_df = pd.read_csv(processed / "test.csv")
test_df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,26,1.631856,110.804337,yes,yes,3,3,Sometimes,no,3,no,0,0,Sometimes,Public_Transportation,Obesity_Type_III
1,Female,19,1.609495,42.656563,no,yes,3,3,Sometimes,no,3,yes,2,1,no,Public_Transportation,Insufficient_Weight
2,Female,19,1.573816,39.850137,no,yes,1,2,Sometimes,no,1,no,1,2,Sometimes,Public_Transportation,Insufficient_Weight
3,Female,22,1.500000,63.652330,yes,no,2,1,Sometimes,no,2,no,0,1,no,Public_Transportation,Overweight_Level_II
4,Female,27,1.550000,63.000000,no,yes,2,3,Sometimes,no,1,no,0,1,Sometimes,Automobile,Overweight_Level_I


In [5]:
with open(processed / "metadata.json") as f:
    metadata = json.load(f)

print(metadata)

{'num_cols': ['TUE', 'FAF', 'CH2O', 'FCVC', 'Age', 'Height', 'Weight', 'NCP'], 'cat_cols': ['CAEC', 'CALC', 'MTRANS'], 'bin_cols': ['Gender', 'family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC'], 'target_col': 'NObeyesdad'}


In [6]:
le = LabelEncoder()
X = train_df.drop(columns=metadata["target_col"])
y = le.fit_transform(train_df[metadata["target_col"]]).astype(np.int32)

print("Classes:", le.classes_)

Classes: ['Insufficient_Weight' 'Normal_Weight' 'Obesity_Type_I' 'Obesity_Type_II'
 'Obesity_Type_III' 'Overweight_Level_I' 'Overweight_Level_II']


In [7]:
processor = make_processor(metadata)
n_num_cols = len(metadata["num_cols"])

In [8]:
best_trial = fit_xrfm_cross_validation(
    X, y, processor, n_num_cols, task=cfg["task"], device=device
)

print(f"Best params: {best_trial.params}")
print(f"Best macro F1: {best_trial.value}")

[I 2026-04-22 21:32:00,701] A new study created in memory with name: no-name-f1854e33-6a4a-45dc-9bf8-d75542452791


None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 1350, d: 31, and nval: 338
Using cheap batch size
Optimal M batch size: 1350


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]


...

Time taken for round 4: 0.026616573333740234 seconds
Tree has no split, stopping training
Using hard routing for tree prediction
None
Fitting xRFM with 1 trees and 0 iterations per tree



Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 1351, d: 31, and nval: 337
Using expensive batch size
Optimal M batch size: 1351
Time taken for round 0: 0.025955677032470703 seconds
Using expensive batch size
Optimal M batch size: 1351
Time taken for round 1: 0.027306079864501953 seconds
Using expensive batch size
Optimal M batch size: 1351
Time taken for round 2: 0.026728153228759766 seconds
Using expensive batch size
Optimal M batch size: 1351
Time taken for round 3: 0.026517629623413086 seconds
Using expensive batch size
Optimal M batch size: 1351
Time taken for round 4: 0.02618408203125 seconds


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]
[I 2026-04-22 21:33:27,345] Trial 99 finished with value: 0.9764840008382087 and parameters: {'exponent': 0.9991570782873668, 'kernel_type': 'kpq', 'bandwidth': 2.254931842128807, 'diag': True, 'norm_p': 1.020633350784882, 'reg': 0.017740426487912565}. Best is trial 43 with value: 0.9821941233169648.


Tree has no split, stopping training
Using hard routing for tree prediction
Best params: {'exponent': 0.8266944178149827, 'kernel_type': 'kpq', 'bandwidth': 1.8169724544596653, 'diag': True, 'norm_p': 0.8739867756221869, 'reg': 5.7741155858099045e-05}
Best macro F1: 0.9821941233169648


In [9]:
model, pipeline, fit_time, train_metrics, val_metrics = fit_best_xrfm(
    X,
    y,
    best_trial,
    processor,
    n_num_cols,
    task=cfg["task"],
    device=device,
)

print(f"Training time: {fit_time:.2f}s")
print(
    f"Train Accuracy: {train_metrics['accuracy']:.4f}  AUC-ROC: {train_metrics['auc_roc']:.4f}"
)
print(
    f"Val Accuracy: {val_metrics['accuracy']:.4f}  AUC-ROC: {val_metrics['auc_roc']:.4f}"
)

joblib.dump(model, MODELS_DIR / data_name / "xrfm.joblib")
joblib.dump(pipeline, MODELS_DIR / data_name / "xrfm_pipeline.joblib")

None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 1350, d: 31, and nval: 338
Using expensive batch size
Optimal M batch size: 1350
Time taken for round 0: 0.02791118621826172 seconds
Using expensive batch size
Optimal M batch size: 1350
Time taken for round 1: 0.028059959411621094 seconds
Using expensive batch size
Optimal M batch size: 1350
Time taken for round 2: 0.027932405471801758 seconds
Using expensive batch size
Optimal M batch size: 1350
Time taken for round 3: 0.027331113815307617 seconds
Using expensive batch size
Optimal M batch size: 1350


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Time taken for round 4: 0.02706313133239746 seconds
Tree has no split, stopping training
Using hard routing for tree prediction
Using hard routing for tree prediction
Training time: 0.16s
Train Accuracy: 1.0000  AUC-ROC: 1.0000
Val Accuracy: 0.9882  AUC-ROC: 0.9997


['/home/shado/college_project/comp9417/9417GroupProject/models/obesity_level/xrfm_pipeline.joblib']

In [10]:
model = joblib.load(MODELS_DIR / data_name / "xrfm.joblib")
pipeline = joblib.load(MODELS_DIR / data_name / "xrfm_pipeline.joblib")

In [11]:
X_test = test_df.drop(columns=metadata["target_col"])
y_test = le.transform(test_df[metadata["target_col"]]).astype(np.int32)

X_test = pipeline.transform(X_test)
t0 = time.perf_counter()
y_proba = model.predict_proba(X_test)
infer_time = (time.perf_counter() - t0) / len(X_test)

Using hard routing for tree prediction


In [12]:
results = evaluate_classification(y_test, y_proba)
print(f"Accuracy:  {results['accuracy']:.4f}")
print(f"AUC-ROC:   {results['auc_roc']:.4f}")
print(f"Inference time per sample: {infer_time * 1e6:.2f}µs")

Accuracy:  0.9835
AUC-ROC:   0.9997
Inference time per sample: 7.32µs


In [13]:
output = {
    **results,
    "train_metrics": train_metrics,
    "val_metrics": val_metrics,
    "fit_time_s": fit_time,
    "infer_time_per_sample_s": infer_time,
    "best_params": {
        "tuning_metric": model.tuning_metric,
        "split_method": model.split_method,
        "max_leaf_size": model.max_leaf_size,
        **model.rfm_params,
    },
}

results_dir = RESULTS_BENCHMARK / "obesity_level"
with open(results_dir / "xrfm.json", "w") as f:
    json.dump(output, f, indent=4)

plot_roc_curve(
    y_test,
    y_proba,
    save_path=results_dir / "roc_curve_xrfm.png",
    class_names=list(le.classes_),
)
plot_confusion_matrix(
    y_test,
    y_proba,
    save_path=results_dir / "confusion_matrix_xrfm.png",
    class_names=list(le.classes_),
)
save_classification_report(
    y_test,
    y_proba,
    save_path=results_dir / "classification_report_xrfm.json",
    class_names=list(le.classes_),
)

print(f"Results saved to {results_dir / 'xrfm.json'}")

Results saved to /home/shado/college_project/comp9417/9417GroupProject/results/obesity_level/xrfm.json
